# Using the Claude and Codex models

The live check for spec 0012: one call per model, through its controller, on a
one-line passage with the purpose/risk prompt. Each call runs on your
subscription (Claude Code's login, Codex's ChatGPT login), never an API key,
and spends a little of its quota: about 1.6k input tokens for Claude, about
11k for Codex (its tool definitions).

Settings come from `configs/claude.toml` and `configs/codex.toml`; pass an
argument to `ClaudeModel(...)` / `CodexModel(...)` to override one.

In [ ]:
from engine.controllers.claude_controller import ClaudeController
from engine.controllers.codex_controller import CodexController
from engine.models.base_model import Prompt
from engine.models.claude_model import ClaudeModel
from engine.models.codex_model import CodexModel
from engine.repositories.prompts import loader as prompts

spec = prompts.load("purpose_risk")
passage = "supports healthy joints. do not take if pregnant. keep out of reach of children."
prompt = Prompt(system=spec.system, user=passage, stop=spec.stop, max_tokens=spec.max_tokens)

## Ready

No model call: the binary runs and the CLI is logged in. A failure names the
fix -- `claude login`, `codex login`, or the toml line whose binary moved
(Codex's moves when VSCode updates its extension).

In [ ]:
claude = ClaudeController()
codex = CodexController()
claude.model.ready()
codex.model.ready()
{name: (m.binary, m.model, m.effort) for name, m in [("claude", claude.model), ("codex", codex.model)]}

## One call each

Expected from both: `P: supports healthy joints` / `R: do not take if
pregnant` / `END`, `ok=True` on the first attempt. `Generation.model` says
which model and CLI version answered.

In [ ]:
for name, controller in [("claude", claude), ("codex", codex)]:
    [outcome] = controller.handle([prompt], checks=spec.checks)
    g = outcome.generation
    print(f"== {name}: ok={outcome.ok} attempts={outcome.attempts} failed={outcome.failed}")
    print(f"   model={g.model} latency={g.latency_ms} ms usage={g.usage} stop={g.stop_reason!r}")
    print(g.text)

## Close

Cancels anything queued and kills running calls. Safe to run twice.

In [ ]:
claude.close()
codex.close()